In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import datetime
import sys
# %matplotlib notebook
%matplotlib inline

In [ ]:
matplotlib.__version__

In [ ]:
def get_timedelta(duration_h_m_s):
    try:
        t_parse = datetime.datetime.strptime(duration_h_m_s,"%H:%M:%S")
    except:
        sys.stderr.write("Error parsing: '{}'".format(duration_h_m_s))
        raise
    t_delta = datetime.timedelta(hours=t_parse.hour, minutes=t_parse.minute, seconds=t_parse.second)
    return t_delta

In [ ]:
df_tsv = pd.read_csv("Fellowship of the Ring - page-to-timestamp.tsv",sep='\t')

In [ ]:
df_tsv

In [ ]:
df_chapters = pd.read_csv("Fellowship of the Ring - chapters.tsv",sep='\t')
df_chapters

In [ ]:
def get_duration_s(maybe_str):
    if maybe_str is not np.nan:
        t_delta = get_timedelta(maybe_str)
        t_s = t_delta.total_seconds()
    else:
        t_s = np.nan
    return t_s
page_number_raw = df_tsv['Cumulative page number']
t_start_str = df_tsv['Movie start']
t_stop_str = df_tsv['Movie stop']
page_number_l = []
t_start_l = []
t_stop_l = []
for page, t1, t2 in zip(page_number_raw, t_start_str, t_stop_str):
    page_number_l.append(page)
    t_start_l.append(get_duration_s(t1))
    t_stop_l.append(get_duration_s(t2))
page_number = np.array(page_number_l)
t_start = np.array(t_start_l)
t_stop = np.array(t_stop_l)
t_mid = (t_start + t_stop)/2
df_plot = pd.DataFrame(data={
    'page_number':page_number,
    't_start':t_start,
    't_stop':t_stop,
    't_mid':t_mid,
})

In [ ]:
df_tsv['t_mid'] = t_mid
df_tsv['page'] = df_tsv['Cumulative page number']

In [ ]:
page_number = df_tsv['Cumulative page number']
t_start = pd.to_datetime(df_tsv['Movie start'], format='%H:%M:%S').dt.time
t_stop = pd.to_datetime(df_tsv['Movie stop'], format='%H:%M:%S').dt.time
# t_interval = pd.Interval(left=t_start, right=t_stop, closed='both')
# t_mid = t_interval.mid
# df_plot = pd.DataFrame(data={
#     'page_number':page_number,
#     't_start':t_start,
#     't_stop':t_stop,
#     't_mid':t_mid,
# })

In [ ]:
chapter_page_numbers = df_chapters.Page.to_list()

In [ ]:
t_start.shape, df_plot.page_number.shape

In [ ]:
def format_timedelta(x, pos):
    # str(datetime.timedelta) is HH:MM:SS
    return str(datetime.timedelta(seconds=int(x)))

In [ ]:
class QuoteInfo(object):
    def __init__(self, page, text, xy, xy_text, fontsize):
        self.page = page
        self.text = text
        self.xy = xy
        self.xy_text = xy_text
        self.fontsize = fontsize
    def __repr__(self):
        return self.__class__.__name__ + '(' + str(list(self.__dict__.keys())) + ')'
    def __str__(self):
        return self.__class__.__name__ + '(' + str(list(self.__dict__.keys())) + ')'

chosen_i = (2,45,62,74,91)
quotes = {}
for i in chosen_i:
    x = df_tsv.iloc[i].t_mid
    y = df_tsv.iloc[i].page
    quotes[i] = QuoteInfo(
        df_tsv.iloc[i].page,
        df_tsv.iloc[i]["Movie quote / description"],
        (x, y),
        (x + 800, y + 4),
        8,
    )

In [ ]:
fig, ax = plt.subplots(constrained_layout=True, figsize=(6,13), dpi=150)
ax.plot(df_plot.t_mid, df_plot.page_number, '.', color="forestgreen", fillstyle="none", markersize=10);
# yfmt = matplotlib.dates.DateFormatter('%H:%M:%S')
# ax.yaxis.set_major_formatter(yfmt)
# fig.autofmt_ydate();
for index, row in df_chapters.iterrows():
    ax.axhline(y=row.Page, linestyle='--', color='lightgray', zorder=0)
    if row.Book < 2:
        this_x = 12500
        horizontalalignment='right'
    else:
        this_x = 0
        horizontalalignment='left'
    ax.text(this_x, row.Page+6, row.Title, horizontalalignment=horizontalalignment)
    # print(index, row.Title, row.Page)
# quotes = [
#     2: ('"Hobbits must seem of little importance"', (df_tsv.iloc[2].t_mid+, df_tsv.iloc[2].page), (2000,df_tsv.iloc[2].page+4), 8),
#     ('"Keep it secret. Keep it safe."', (2050, 39), (2000+500, 39+4), 8),
#     ('"I wish the Ring had never come to me."', (7740-150, 55), (7740-5100, 55-3), 7),
#     ('"Many that live deserve death,\nand some that die, deserve life.\nCan you give it to them, Frodo?"', (7850, 65), (8500,65+6), 7),
#     ('"It\'s a dangerous business going out your door."', (2690+150, 82-1), (2690+800, 82-4), 7),
#     ('"Round here, he\'s known as Strider."', (3650+150, 177), (3650+900, 177+4), 7),
#     ("The Witch King stabs Frodo", (4425+150,221), (4425+800,221-4), 8),
#     ('"This is beyond my skill to heal."', (4497-110, 230), (4497-4300, 230-4), 7),
#     ('"I\'ve no memory of this place."', (7610+150, 350), (7610+800, 350-4), 8),
#     ('"You cannot pass!"', (8770+150, 370), (8770+800, 370-3), 8),
# ]
for quote in quotes:
    ax.annotate(
        this_text,
        xy=this_xy,
        xytext=this_xytext,
        arrowprops=dict(
            width=0.2,
            headwidth=4,
            headlength=4,
            color='black',
    #         arrowstyle="-|>,head_width=0.4,head_length=0.8",
    #         shrinkA=0,
    #         shrinkB=0
        ),
        color='black',
        fontsize=this_fontsize,
        xycoords='data',
        textcoords='data',
    #     transform = ax.transAxes,
    )
ax.set_ylabel("Page number of book")
ax.set_xlabel("Duration into movie");
ax.xaxis.set_label_position('top');
ax.xaxis.tick_top()
ax.xaxis.set_major_formatter(
     matplotlib.ticker.FuncFormatter(format_timedelta)
)
ax.set_yticks(chapter_page_numbers)
ax.set_ylim(1, 475)
ax.yaxis.set_inverted(True)
ax.set_title("The Fellowship of the Ring");

In [ ]:
fig.canvas.draw();

In [ ]:
# Save to file
fig.savefig(
    "fellowship-of-the-ring.png",
    bbox_inches='tight',
    metadata = {"Title": "example title", "Author": "Firstname Lastname"},
    dpi=200,
    facecolor="w", # white background
);
# Save to file
fig.savefig(
    "fellowship-of-the-ring.svg",
    bbox_inches='tight',
    metadata = {"Title": "example title", "Author": "Firstname Lastname"},
    facecolor="w", # white background
);

In [ ]:
plt.close(fig); del fig, ax;

TODO:

- [ ] Add vertical lines for chapter breaks

In [ ]:
import sys; sys.exit(1)


## Emma messing around

In [ ]:
# !pip install plotly
# !pip install pyjanitor
from plotly import express, offline
from plotly import graph_objects as go
from janitor import clean_names
import time 

In [ ]:
offline.init_notebook_mode()

In [ ]:
lotr = clean_names(pd.read_csv("Fellowship of the Ring - page-to-timestamp.tsv",sep='\t'))

In [ ]:
lotr

In [ ]:
lotr.info()

In [ ]:
def clean_hhmmss(tm):
    if tm is None:
        return pd.NaT
    elif type(tm) == float:
        return pd.NaT 
    else:
        return pd.to_datetime(tm, errors = 'coerce')

In [ ]:
def get_midpoint(start, stop):
    if (start is None or stop is None or type(start) == float or type(stop) == float):
        return pd.NaT
    else:
        clean_start = clean_hhmmss(min(start,stop))
        clean_stop = clean_hhmmss(max(stop, stop))
        duration = clean_stop - clean_start
        midpoint = clean_start + (duration/2)
        return midpoint.strftime('%H:%M:%S')
    
    

In [ ]:
clean_hhmmss(lotr['movie_start'][2])

In [ ]:
# lotr.assign(
#     clean_movie_start = np.where(pd.isna(lotr['movie_start']), pd.NaT, pd.to_datetime(lotr['movie_start'], '%H:%M:%S'))
# )

In [ ]:
lotr['clean_movie_start'] = lotr.apply(lambda row: clean_hhmmss(row['movie_start']), axis = 1)
lotr['clean_movie_stop'] = lotr.apply(lambda row: clean_hhmmss(row['movie_stop']), axis = 1)

In [ ]:
lotr = lotr.assign(
    duration = lotr['clean_movie_stop'] - lotr['clean_movie_start']
)

In [ ]:
lotr['midpoint'] = lotr.apply(lambda row: get_midpoint(row['movie_start'],row['movie_stop']), axis = 1)
lotr['midpoint_str'] = lotr['midpoint'].to_string()

In [ ]:
lotr

In [ ]:
lotr.info()

In [ ]:
express.scatter(
    lotr,
    x = 'book_page_number',
    y = 'movie_start'
)

In [ ]:
express.scatter(
    lotr,
    x = 'book_page_number',
    y = 'midpoint'
).update_yaxes(tickformat="%H:%M:%S")

In [ ]:
express.scatter(
    lotr,
    x = 'book_page_number',
    y = 'midpoint_str'
)